# Advanced Web Vulnerability Scanner

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import socket
import requests
from urllib.parse import urlparse
import threading
!pip install dnspython
from geopy.geocoders import Nominatim
from bs4 import BeautifulSoup

# Function to check SSL
def check_ssl(url):
    if url.startswith("https://"):
        return True
    else:
        return False

# Function to check for common open ports with service versioning
def check_ports(host):
    open_ports = []
    common_ports = [80, 443, 8080, 8000, 22, 21]
    for port in common_ports:
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(1)
            result = sock.connect_ex((host, port))
            if result == 0:
                service_version = get_service_version(host, port)
                open_ports.append((port, service_version))
            sock.close()
        except socket.error:
            continue
    return open_ports

# Function to get service version from open port
def get_service_version(host, port):
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(1)
        sock.connect((host, port))
        banner = sock.recv(1024).decode().strip()
        sock.close()
        return banner if banner else "Unknown Version"
    except Exception:
        return "Unknown Version"

# Function to check for SQL Injection
def check_sql_injection(url):
    payloads = ["' OR 1=1 --", "' OR 'a'='a", "' UNION SELECT NULL, NULL --"]
    vulnerable = []
    for payload in payloads:
        response = requests.get(url + payload)
        if "error" in response.text or "warning" in response.text:
            vulnerable.append(payload)
    return vulnerable

# Function to check for XSS
def check_xss(url):
    payloads = ["<script>alert('XSS')</script>", "<img src='x' onerror='alert(1)'>"]
    vulnerable = []
    for payload in payloads:
        response = requests.get(url + payload)
        if payload in response.text:
            vulnerable.append(payload)
    return vulnerable

# Function to check HTTP headers for missing security headers
def check_headers(url):
    headers_to_check = [
        "Strict-Transport-Security",
        "Content-Security-Policy",
        "X-Content-Type-Options",
        "X-Frame-Options",
        "X-XSS-Protection"
    ]
    try:
        response = requests.get(url)
        missing_headers = [header for header in headers_to_check if header not in response.headers]
        return missing_headers
    except requests.exceptions.RequestException:
        return ["Request failed"]

# Function to perform DNS resolution and geolocation lookup
def dns_lookup_and_geolocate(url):
    try:
        parsed_url = urlparse(url)
        hostname = parsed_url.hostname
        resolver = dns.resolver.Resolver()
        ip = resolver.resolve(hostname, 'A')[0].to_text()

        geolocator = Nominatim(user_agent="geoapi")
        location = geolocator.geocode(ip)
        return ip, location
    except Exception as e:
        return "Unable to resolve", None

# Function to scan the website
def scan_website(url):
    print(f"\nScanning {url}...\n")

    ssl = check_ssl(url)
    print(f"SSL Enabled: {'Yes' if ssl else 'No'}")

    parsed_url = urlparse(url)
    host = parsed_url.hostname
    open_ports = check_ports(host)
    print(f"Open Ports and Services: {open_ports}")

    missing_headers = check_headers(url)
    if missing_headers:
        print(f"Missing Security Headers: {', '.join(missing_headers)}")
    else:
        print("No critical security headers missing.")

    sql_vulnerabilities = check_sql_injection(url)
    if sql_vulnerabilities:
        print(f"Potential SQL Injection Vulnerabilities: {', '.join(sql_vulnerabilities)}")

    xss_vulnerabilities = check_xss(url)
    if xss_vulnerabilities:
        print(f"Potential XSS Vulnerabilities: {', '.join(xss_vulnerabilities)}")

    ip, location = dns_lookup_and_geolocate(url)
    print(f"IP Address: {ip}")
    if location:
        print(f"Geolocation: {location.address}")
    else:
        print("Geolocation lookup failed.")

# Example usage
url = input("Enter a URL to scan (e.g. https://example.com): ")
scan_website(url)
